# Phase 2 — Feature Engineering

This notebook demonstrates the complete feature engineering workflow using the production modules inside `src/fx_forecast/features`.

In [1]:
import pandas as pd

from fx_forecast.config.settings import settings
from fx_forecast.config.paths import PROCESSED_DATA_DIR
from fx_forecast.features.calendar import create_calendar_features
from fx_forecast.features.statistical import create_statistical_features
from fx_forecast.features.technical import create_technical_features
from fx_forecast.features.target import create_target_features
from fx_forecast.features.selection import select_features
from fx_forecast.features.pipeline import run_feature_pipeline

## Load processed dataset

In [2]:
symbol = settings.currency_pairs[0].replace("=", "_")
df = pd.read_csv(
    PROCESSED_DATA_DIR / f"{symbol}.csv",
    index_col=0,
    parse_dates=True,
)
df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,1.209863,1.209863,1.209863,1.209863,0
2015-01-02,1.208941,1.208956,1.201080,1.208868,0
2015-01-05,1.194643,1.197590,1.188909,1.195500,0
2015-01-06,1.193902,1.197000,1.188693,1.193830,0
2015-01-07,1.187536,1.190000,1.180401,1.187479,0


## Dataset overview

In [3]:
df.info()
df.describe().T

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3015 entries, 2015-01-01 to 2026-08-03
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3015 non-null   float64
 1   High    3015 non-null   float64
 2   Low     3015 non-null   float64
 3   Open    3015 non-null   float64
 4   Volume  3015 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 141.3 KB


,count,mean,std,min,25%,50%,75%,max
Close,3015.0,1.122249,0.051748,0.959619,1.086195,1.120599,1.163196,1.251001
High,3015.0,1.125778,0.051484,0.967006,1.089452,1.124099,1.165902,1.255808
Low,3015.0,1.118679,0.051940,0.954016,1.082784,1.116807,1.160174,1.245051
Open,3015.0,1.122233,0.051742,0.959619,1.086154,1.120498,1.163291,1.251267
Volume,3015.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Calendar features

In [4]:
calendar_df = create_calendar_features(df)
calendar_df.filter(regex="day|week|month|quarter|year").head()

2026-08-04 13:36:27 | SUCCESS  | fx_forecast.features.calendar:create_calendar_features:45 | Calendar features created.


,day_of_week,day_of_month,day_of_year,week_of_year,month,quarter,year,is_month_start,is_month_end,is_quarter_start,is_quarter_end,is_year_start,is_year_end
Date,,,,,,,,,,,,,
2015-01-01,3,1,1,1,1,1,2015,1,0,1,0,1,0
2015-01-02,4,2,2,1,1,1,2015,0,0,0,0,0,0
2015-01-05,0,5,5,2,1,1,2015,0,0,0,0,0,0
2015-01-06,1,6,6,2,1,1,2015,0,0,0,0,0,0
2015-01-07,2,7,7,2,1,1,2015,0,0,0,0,0,0


## Statistical features

In [5]:
stats_df = create_statistical_features(calendar_df)
stats_df.filter(regex="return|rolling").head()

2026-08-04 13:37:24 | SUCCESS  | fx_forecast.features.statistical:create_statistical_features:52 | Statistical features created.


,return,log_return,rolling_mean_5,rolling_std_5,rolling_min_5,rolling_max_5,rolling_mean_10,rolling_std_10,rolling_min_10,rolling_max_10,rolling_mean_20,rolling_std_20,rolling_min_20,rolling_max_20
Date,,,,,,,,,,,,,,
2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,-0.000762,-0.000762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,-0.011827,-0.011897,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-06,-0.000621,-0.000621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,-0.005332,-0.005346,1.198977,0.009915,1.187536,1.209863,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Technical indicators

In [6]:
technical_df = create_technical_features(stats_df)
technical_df.filter(regex="sma|ema").head()

2026-08-04 13:38:01 | SUCCESS  | fx_forecast.features.technical:create_technical_features:62 | Technical indicators created.


,sma_5,sma_10,sma_20,ema_5,ema_10,ema_20
Date,,,,,,
2015-01-01,NaN,NaN,NaN,1.209863,1.209863,1.209863
2015-01-02,NaN,NaN,NaN,1.209556,1.209695,1.209775
2015-01-05,NaN,NaN,NaN,1.204585,1.206959,1.208334
2015-01-06,NaN,NaN,NaN,1.201024,1.204585,1.206959
2015-01-07,1.198977,NaN,NaN,1.196528,1.201485,1.205110


## Target engineering

In [7]:
target_df = create_target_features(technical_df)
target_df.filter(regex="target|future_return").head()

2026-08-04 13:38:19 | SUCCESS  | fx_forecast.features.target:create_target_features:48 | Target features created.


,target,target_3d,target_5d,future_return_1d,future_return_3d,future_return_5d
Date,,,,,,
2015-01-01,1.208941,1.193902,1.183600,-0.000762,-0.013193,-0.021707
2015-01-02,1.194643,1.187536,1.179607,-0.011827,-0.017706,-0.024265
2015-01-05,1.193902,1.183600,1.187070,-0.000621,-0.009244,-0.006339
2015-01-06,1.187536,1.179607,1.183152,-0.005332,-0.011973,-0.009004
2015-01-07,1.183600,1.187070,1.177829,-0.003314,-0.000392,-0.008174


## Feature selection

In [8]:
clean_df = target_df.dropna()

X, y = select_features(clean_df)

print("Feature matrix:", X.shape)
print("Target vector :", y.shape)

X.head()

2026-08-04 13:38:34 | SUCCESS  | fx_forecast.features.selection:select_features:48 | Selected 43 feature(s).
Feature matrix: (2991, 43)
Target vector : (2991,)


,Close,High,Low,Open,Volume,day_of_week,day_of_month,day_of_year,week_of_year,month,...,sma_10,sma_20,ema_5,ema_10,ema_20,target_3d,target_5d,future_return_1d,future_return_3d,future_return_5d
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-28,1.136738,1.138200,1.130780,1.136699,0,2,28,28,5,1,...,1.148369,1.169492,1.135297,1.146440,1.163010,1.130902,1.145738,-0.007558,-0.005134,0.007917
2015-01-29,1.128146,1.136800,1.126405,1.128477,0,3,29,29,5,1,...,1.143324,1.165406,1.132914,1.143114,1.159690,1.134173,1.131875,0.004795,0.005342,0.003305
2015-01-30,1.133556,1.136400,1.128599,1.133735,0,4,30,30,5,1,...,1.140306,1.161636,1.133128,1.141376,1.157201,1.145738,1.147197,-0.002341,0.010747,0.012034
2015-02-02,1.130902,1.136000,1.129293,1.130467,0,0,2,33,6,2,...,1.137736,1.158449,1.132386,1.139472,1.154696,1.131875,1.131439,0.002892,0.000860,0.000475
2015-02-03,1.134173,1.145001,1.131790,1.133697,0,1,3,34,6,2,...,1.135173,1.155463,1.132981,1.138508,1.152742,1.147197,1.132695,0.010197,0.011483,-0.001303


## Execute the production feature pipeline

In [ ]:
X_pipeline, y_pipeline = run_feature_pipeline(df)

print(X_pipeline.shape)
print(y_pipeline.shape)

X_pipeline.head()

## Inspect target

In [ ]:
y_pipeline.head()

# Phase 2 Summary

The production feature engineering pipeline performs:

1. Calendar feature generation
2. Statistical feature generation
3. Technical indicator generation
4. Target engineering
5. Missing-value removal
6. Feature selection
7. Returns a production-ready feature matrix (X) and target vector (y)

The outputs are ready for Phase 3 (EDA) and Phase 4 (Model Development).